## ERP Locations Silver Ttransformation

### 00. Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col
from datetime import datetime

### 01. Build today's folder name and read from bronze table

In [0]:
today = datetime.today().strftime("%Y_%m_%d")   # e.g. 2026_06_23
bronze_table = f"`abc_business-core`.bronze_{today}.erp_locations"
df = spark.table(bronze_table)

### 02. Trimmming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### 03. Customer ID Cleanup

In [0]:
df = df.withColumn("cid", F.regexp_replace(col("cid"), "-", ""))

### 04. Country Normalization


In [0]:
df = df.withColumn(
    "cntry",
    F.when(col("cntry") == "DE", "Germany")
     .when(col("cntry").isin("US", "USA"), "United States")
     .when((col("cntry") == "") | col("cntry").isNull(), "n/a")
     .otherwise(col("cntry"))
)

### 05. Renaming Columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

### 06. Sanity checks of dataframe

In [0]:

df.limit(5).display()

### 07. Writing Silver Table

In [0]:
table_name = f"`abc_business-core`.silver.erp_locations_{today}"
df.write.mode("overwrite").format("delta").saveAsTable(table_name)

### 08. Sanity checks of silver table

In [0]:
df = spark.sql(f"SELECT * FROM {table_name} LIMIT 5")
df.show()